# LeetCode 3190 — Find Minimum Operations to Make All Elements Divisible by Three bằng Python

Link đề: https://leetcode.com/problems/find-minimum-operations-to-make-all-elements-divisible-by-three/ · Độ khó: **Easy** · Chủ đề: Array, Math

Skeleton LeetCode:

```python
from typing import List


class Solution:
    def minimumOperations(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và ví dụ được chép/dịch theo đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code chạy thật.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi thao tác **cộng hoặc trừ 1** cho **một** phần tử bất kỳ; cần số thao tác **tối thiểu** để **mọi** phần tử chia hết cho `3`.
2. Nắm các công cụ Python: toán tử `%` (số dư), `x % 3 == 0` (chia hết), `min(a, b)`, `sum(...)` trên biểu thức, **`True` tính như `1`**, `enumerate`/vòng `for`, `abs`, và phép chia nguyên `//`.
3. Đi từ cách **thử từng khoảng dịch `d = 0, 1, 2...` cho mỗi phần tử** → **công thức `min(r, 3 - r)`** → **chỉ đếm phần tử không chia hết cho 3** → **đếm theo lớp số dư** → **tìm bội của 3 gần nhất**.
4. Hiểu mấu chốt: số dư khi chia `3` chỉ là `0`, `1` hoặc `2`; số dư `1` cần **trừ 1**, số dư `2` cần **cộng 1**; nên mỗi phần tử **không chia hết cho 3 tốn đúng 1 thao tác**, phần tử chia hết tốn `0`.
5. Tránh các lỗi kinh điển: cộng `x % 3` (số dư `2` bị tính thành `2` thao tác), chỉ cho phép trừ (bỏ quên cộng), đếm nhầm các phần tử **chia hết** (ngược điều kiện), và nghĩ rằng một thao tác tác động lên nhiều phần tử.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Trong **một thao tác** bạn có thể **cộng hoặc trừ `1`** cho **bất kỳ một** phần tử của `nums`.

Trả về **số thao tác tối thiểu** để **tất cả** phần tử của `nums` chia hết cho `3`.

**Ví dụ 1**

- Input: `nums = [1,2,3,4]`
- Output: `3`
- Giải thích: dùng `3` thao tác: trừ `1` ở số `1` (thành `0`), cộng `1` ở số `2` (thành `3`), trừ `1` ở số `4` (thành `3`). Số `3` đã chia hết.

**Ví dụ 2**

- Input: `nums = [3,6,9]`
- Output: `0`
- Giải thích: mọi phần tử đã chia hết cho `3`.

**Ràng buộc:** `1 <= nums.length <= 50`, `1 <= nums[i] <= 50`.

In [1]:
import random
from collections import Counter
from typing import List

examples = [
    ([1, 2, 3, 4], 3),
    ([3, 6, 9], 0),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[1, 2, 3, 4] -> kỳ vọng 3
[3, 6, 9] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Số dư và chia hết: `%`

`x % 3` là số dư khi chia `x` cho `3`, chỉ có thể là `0`, `1` hoặc `2`. `x` chia hết cho `3` **khi và chỉ khi** `x % 3 == 0`.

In [2]:
for x in range(1, 8):
    print(x, "% 3 =", x % 3, "| chia hết cho 3:", x % 3 == 0)
# 1->1, 2->2, 3->0 (chia hết), 4->1, 5->2, 6->0 (chia hết), 7->1

1 % 3 = 1 | chia hết cho 3: False
2 % 3 = 2 | chia hết cho 3: False
3 % 3 = 0 | chia hết cho 3: True
4 % 3 = 1 | chia hết cho 3: False
5 % 3 = 2 | chia hết cho 3: False
6 % 3 = 0 | chia hết cho 3: True
7 % 3 = 1 | chia hết cho 3: False


### 3.2. Mỗi thao tác đổi số dư thế nào?

Cộng `1` vào `x` thì số dư (mod 3) tăng `1` (từ `2` thì quay về `0`); trừ `1` thì giảm `1` (từ `0` thì quay về `2`). Vì thế:

- `x % 3 == 1`: **trừ 1** → số dư thành `0` (1 thao tác); hoặc cộng 2 lần (2 thao tác).
- `x % 3 == 2`: **cộng 1** → số dư thành `0` (1 thao tác); hoặc trừ 2 lần (2 thao tác).

In [3]:
for x in [4, 5]:
    print(f"x={x}, dư {x % 3}:  x-1 = {x - 1} (dư {(x - 1) % 3}),  x+1 = {x + 1} (dư {(x + 1) % 3})")
# x=4 (dư 1): trừ 1 được 3 (dư 0).  x=5 (dư 2): cộng 1 được 6 (dư 0).

x=4, dư 1:  x-1 = 3 (dư 0),  x+1 = 5 (dư 2)
x=5, dư 2:  x-1 = 4 (dư 1),  x+1 = 6 (dư 0)


### 3.3. `min(a, b)` và `sum` trên biểu thức

`min(a, b)` trả số nhỏ hơn. `sum(biểu_thức for x in nums)` cộng giá trị biểu thức của từng phần tử; rất gọn cho "tổng chi phí của từng phần tử".

In [4]:
print(min(1, 2), min(2, 3 - 2))                      # 1 1
nums = [1, 2, 3, 4]
print([min(x % 3, 3 - x % 3) for x in nums])         # [1, 1, 0, 1]   chi phí từng phần tử
print(sum(min(x % 3, 3 - x % 3) for x in nums))      # 3

1 1
[1, 1, 0, 1]
3


### 3.4. `True` tính như `1` — đếm bằng `sum`

Trong Python `True == 1` và `False == 0`, nên `sum(điều_kiện for x in nums)` **đếm** số phần tử thoả điều kiện.

In [5]:
nums = [1, 2, 3, 4]
print(True + True, False + 5)                        # 2 5
print(sum(x % 3 != 0 for x in nums))                 # 3  (1, 2, 4 không chia hết cho 3)
print(sum(x % 3 == 0 for x in nums))                 # 1  (chỉ có 3 chia hết)

2 5
3
1


### 3.5. Đếm theo lớp số dư bằng `Counter` hoặc mảng nhỏ

Có đúng `3` lớp số dư `0, 1, 2`; ta có thể đếm mỗi lớp có bao nhiêu phần tử: `cnt = [0] * 3`, `cnt[x % 3] += 1`.

In [6]:
cnt = [0] * 3
for x in [1, 2, 3, 4]:
    cnt[x % 3] += 1
print(cnt)                                           # [1, 2, 1]  -> một số dư 0, hai số dư 1 (1 và 4), một số dư 2 (số 2)
print(Counter(x % 3 for x in [1, 2, 3, 4]))          # Counter({1: 2, 0: 1, 2: 1})

[1, 2, 1]
Counter({1: 2, 2: 1, 0: 1})


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [1, 2, 3, 4]`, xét **từng phần tử độc lập** (một thao tác chỉ đụng một phần tử, nên chi phí tổng là **tổng chi phí từng phần tử**):

| x | `x % 3` | gần nhất là bội 3 nào? | cần mấy thao tác |
|---|---|---|---|
| 1 | 1 | `0` (trừ 1) | 1 |
| 2 | 2 | `3` (cộng 1) | 1 |
| 3 | 0 | chính nó | 0 |
| 4 | 1 | `3` (trừ 1) | 1 |

Tổng `1 + 1 + 0 + 1 = 3`.

Nhận xét: các bội của `3` cách nhau `3` đơn vị, nên từ một số **bất kỳ** luôn có một bội của `3` cách nó **tối đa `1`** đơn vị. Số không chia hết cho `3` thì bội gần nhất cách nó **đúng `1`** (vì không thể cách `0`). Vậy mỗi số tốn `0` hoặc `1` thao tác.

In [7]:
for x in range(1, 11):
    down = x % 3                         # số bước trừ để về bội 3 bên dưới
    up = (3 - x % 3) % 3                 # số bước cộng để lên bội 3 bên trên
    print(f"x={x:2d}: trừ {down} bước -> {x - down}, cộng {up} bước -> {x + up}, tốn min = {min(down, up)}")

x= 1: trừ 1 bước -> 0, cộng 2 bước -> 3, tốn min = 1
x= 2: trừ 2 bước -> 0, cộng 1 bước -> 3, tốn min = 1
x= 3: trừ 0 bước -> 3, cộng 0 bước -> 3, tốn min = 0
x= 4: trừ 1 bước -> 3, cộng 2 bước -> 6, tốn min = 1
x= 5: trừ 2 bước -> 3, cộng 1 bước -> 6, tốn min = 1
x= 6: trừ 0 bước -> 6, cộng 0 bước -> 6, tốn min = 0
x= 7: trừ 1 bước -> 6, cộng 2 bước -> 9, tốn min = 1
x= 8: trừ 2 bước -> 6, cộng 1 bước -> 9, tốn min = 1
x= 9: trừ 0 bước -> 9, cộng 0 bước -> 9, tốn min = 0
x=10: trừ 1 bước -> 9, cộng 2 bước -> 12, tốn min = 1


## 5. Cách 1 — Brute Force / baseline: thử khoảng dịch `d = 0, 1, 2, ...` cho từng phần tử

**Ý tưởng.** Với mỗi `x`, tăng dần số thao tác `d = 0, 1, 2, ...` cho đến khi `x - d` **hoặc** `x + d` chia hết cho `3`; `d` đầu tiên thoả mãn là chi phí tối thiểu của `x` (vì ta thử từ nhỏ đến lớn). Cộng chi phí các phần tử.

In [8]:
def minimum_operations_search(nums: List[int]) -> int:
    total = 0
    for x in nums:
        d = 0
        while (x - d) % 3 != 0 and (x + d) % 3 != 0:      # chưa có bội của 3 trong tầm d
            d += 1
        total += d                                        # d nhỏ nhất làm x chạm bội của 3
    return total


for nums, expected in examples:
    print(minimum_operations_search(nums), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
0 (kỳ vọng 0 )


**Độ phức tạp (Cách 1).** Mỗi phần tử thử tối đa `2` giá trị `d` (vì luôn có bội 3 trong khoảng cách `1`) ⇒ **Time `O(n)`**, **Space `O(1)`** (hằng số nhỏ). Dù vậy, cách này **làm việc thừa**: ta đã biết trước `d` chỉ là `0` hoặc `1`. Ta có thể tính `d` trực tiếp từ số dư.

## 6. Ý tưởng thuật toán / bước đột phá

Chi phí của `x` chỉ phụ thuộc `r = x % 3`:

- `r = 0`: `0` thao tác.
- `r = 1`: trừ `1` ⇒ `1` thao tác (cộng sẽ cần `2`).
- `r = 2`: cộng `1` ⇒ `1` thao tác (trừ sẽ cần `2`).

Tổng quát `cost(x) = min(r, 3 - r)` cho `r = 1, 2` và `0` cho `r = 0`. Nhưng vì `r ∈ {1, 2}` luôn cho `min(r, 3-r) = 1`, nên:

> **cost(x) = 1 nếu `x % 3 != 0`, ngược lại `0`.**

Đáp án = **số phần tử không chia hết cho `3`**.

In [9]:
nums = [1, 2, 3, 4]
answer = 0
for x in nums:
    cost = 0 if x % 3 == 0 else 1
    answer += cost
    print(f"x={x}: x % 3 = {x % 3}, cost = {cost}, tổng = {answer}")

x=1: x % 3 = 1, cost = 1, tổng = 1
x=2: x % 3 = 2, cost = 1, tổng = 2
x=3: x % 3 = 0, cost = 0, tổng = 2
x=4: x % 3 = 1, cost = 1, tổng = 3


## 7. Các cách tiếp theo

### Cách 2 — Công thức `min(r, 3 - r)` (tổng quát cho mọi `k`)

**Giải thích từng đoạn.**

1. `r = x % 3`: số dư.
2. Trừ `r` đơn vị để xuống bội `k` bên dưới; cộng `3 - r` đơn vị để lên bội bên trên. Chi phí nhỏ hơn là `min(r, 3 - r)`.
3. Với `r = 0`: `min(0, 3) = 0` ✓.

Công thức này đúng cho mọi `k` (thay `3` bằng `k`), nên đáng nhớ.

In [10]:
def minimum_operations_min(nums: List[int]) -> int:
    total = 0
    for x in nums:
        r = x % 3
        total += min(r, 3 - r)           # trừ r bước hoặc cộng (3 - r) bước
    return total


for nums, expected in examples:
    print(minimum_operations_min(nums), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 3 — Đếm phần tử không chia hết cho `3`

**Giải thích.** Vì `min(r, 3 - r) = 1` khi `r != 0`, đáp án chính là số phần tử có `x % 3 != 0`. Một dòng: `sum(x % 3 != 0 for x in nums)`.

In [11]:
def minimum_operations_count(nums: List[int]) -> int:
    return sum(x % 3 != 0 for x in nums)              # True tính là 1


for nums, expected in examples:
    print(minimum_operations_count(nums), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
0 (kỳ vọng 0 )


In [12]:
# Code nộp LeetCode cho Cách 3 (đổi tên lớp thành Solution khi nộp)
class SolutionCount:
    def minimumOperations(self, nums: List[int]) -> int:
        return sum(x % 3 != 0 for x in nums)


print(SolutionCount().minimumOperations([1, 2, 3, 4]))       # 3

3


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 4 — Đếm theo lớp số dư `cnt[0..2]`

**Giải thích.** Đếm số phần tử có số dư `0`, `1`, `2` vào `cnt`. Số dư `1` mỗi phần tử tốn `1`, số dư `2` cũng tốn `1`: đáp án `cnt[1] + cnt[2]`. Cách này tách bạch từng lớp và dễ mở rộng nếu đề đổi giá phí (ví dụ phí thao tác trừ khác phí thao tác cộng — xem biến thể).

In [13]:
def minimum_operations_classes(nums: List[int]) -> int:
    cnt = [0] * 3
    for x in nums:
        cnt[x % 3] += 1                  # cnt[r] = số phần tử có số dư r
    return cnt[1] * 1 + cnt[2] * 1       # mỗi phần tử dư 1 hoặc dư 2 tốn đúng 1 thao tác


for nums, expected in examples:
    print(minimum_operations_classes(nums), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`** (3 ô).

### Cách 5 — Tìm bội của `3` gần nhất rồi đo khoảng cách

**Giải thích.** Bội của `3` gần `x` nhất là `((x + 1) // 3) * 3`: cộng `1` rồi chia nguyên `3` làm "làm tròn" `x/3` đến số nguyên gần nhất (vì số dư `1` làm tròn xuống, số dư `2` làm tròn lên). Chi phí là `abs(x - bội_gần_nhất)`. Cách này nhấn mạnh **ý nghĩa hình học**: khoảng cách từ `x` tới bội `3` gần nhất.

In [14]:
def minimum_operations_nearest(nums: List[int]) -> int:
    total = 0
    for x in nums:
        nearest = ((x + 1) // 3) * 3     # bội của 3 gần x nhất
        total += abs(x - nearest)
    return total


for nums, expected in examples:
    print(minimum_operations_nearest(nums), "(kỳ vọng", expected, ")")
print([((x + 1) // 3) * 3 for x in range(1, 11)])     # [0, 3, 3, 3, 6, 6, 6, 9, 9, 9]

3 (kỳ vọng 3 )
0 (kỳ vọng 0 )
[0, 3, 3, 3, 6, 6, 6, 9, 9, 9]


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`**.

## 8. TRACE từng bước

Ta trace Cách 2 (`min(r, 3 - r)`) trên `nums = [1, 2, 3, 4]` và Cách 1 (thử `d`) trên `nums = [5, 7, 9]`.

Bảng trace Cách 2 với `nums = [1, 2, 3, 4]`:

| i | x | r = `x % 3` | trừ `r` | cộng `3 - r` | cost = min | tổng |
|---|---|---|---|---|---|---|
| 0 | 1 | 1 | 1 | 2 | 1 | 1 |
| 1 | 2 | 2 | 2 | 1 | 1 | 2 |
| 2 | 3 | 0 | 0 | 3 | 0 | 2 |
| 3 | 4 | 1 | 1 | 2 | 1 | 3 |

Đáp án `3`.

In [15]:
def trace_min(nums: List[int]) -> int:
    total = 0
    print(f"nums = {nums}")
    for i, x in enumerate(nums):
        r = x % 3
        down, up = r, 3 - r
        cost = min(down, up)
        total += cost
        print(f"  i={i} x={x} r={r}: trừ {down} / cộng {up} -> cost = {cost}, tổng = {total}")
    return total


assert trace_min([1, 2, 3, 4]) == 3

nums = [1, 2, 3, 4]
  i=0 x=1 r=1: trừ 1 / cộng 2 -> cost = 1, tổng = 1
  i=1 x=2 r=2: trừ 2 / cộng 1 -> cost = 1, tổng = 2
  i=2 x=3 r=0: trừ 0 / cộng 3 -> cost = 0, tổng = 2
  i=3 x=4 r=1: trừ 1 / cộng 2 -> cost = 1, tổng = 3


In [16]:
def trace_search(nums: List[int]) -> int:
    total = 0
    print(f"nums = {nums}")
    for x in nums:
        d = 0
        print(f"  x={x}:", end="")
        while (x - d) % 3 != 0 and (x + d) % 3 != 0:
            print(f" d={d}: ({x - d}, {x + d}) chưa có bội 3;", end="")
            d += 1
        which = "x-d" if (x - d) % 3 == 0 else "x+d"
        print(f" d={d}: {which} = {x - d if which == 'x-d' else x + d} chia hết 3 -> dừng")
        total += d
    print("  tổng =", total)
    return total


assert trace_search([5, 7, 9]) == 2
assert trace_search([1, 1, 1, 3]) == 3

nums = [5, 7, 9]
  x=5: d=0: (5, 5) chưa có bội 3; d=1: x+d = 6 chia hết 3 -> dừng
  x=7: d=0: (7, 7) chưa có bội 3; d=1: x-d = 6 chia hết 3 -> dừng
  x=9: d=0: x-d = 9 chia hết 3 -> dừng
  tổng = 2
nums = [1, 1, 1, 3]
  x=1: d=0: (1, 1) chưa có bội 3; d=1: x-d = 0 chia hết 3 -> dừng
  x=1: d=0: (1, 1) chưa có bội 3; d=1: x-d = 0 chia hết 3 -> dừng
  x=1: d=0: (1, 1) chưa có bội 3; d=1: x-d = 0 chia hết 3 -> dừng
  x=3: d=0: x-d = 3 chia hết 3 -> dừng
  tổng = 3


Với `[5, 7, 9]`: `5` dư `2` ⇒ cộng `1` (thành `6`); `7` dư `1` ⇒ trừ `1` (thành `6`); `9` đã chia hết. Tổng `2`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (đếm phần tử không chia hết cho `3`): ngắn, `O(n)`.

In [17]:
class Solution:
    def minimumOperations(self, nums: List[int]) -> int:
        # mỗi phần tử dư 1 hoặc 2 chỉ cần đúng 1 thao tác (trừ 1 hoặc cộng 1)
        return sum(x % 3 != 0 for x in nums)


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.minimumOperations(nums))

[1, 2, 3, 4] -> 3
[3, 6, 9] -> 0


## 10. Vì sao thuật toán đúng

- **Độc lập từng phần tử:** mỗi thao tác chỉ thay đổi **một** phần tử, nên tổng số thao tác tối thiểu bằng tổng các số thao tác tối thiểu của từng phần tử.
- **Phần tử đã chia hết tốn `0`:** không cần thao tác.
- **Phần tử không chia hết cần ít nhất `1`:** một phần tử không chia hết cho `3` thì không thể giữ nguyên, nên cần `>= 1` thao tác.
- **Một thao tác là đủ:** nếu `x % 3 == 1` thì `x - 1` chia hết cho `3`; nếu `x % 3 == 2` thì `x + 1` chia hết cho `3`. Vậy chi phí tối thiểu của phần tử không chia hết đúng bằng `1`.
- **Kết luận:** đáp án = số phần tử có `x % 3 != 0`. Các cách 2, 4, 5 cho cùng kết quả vì `min(r, 3-r) = 1` với `r ∈ {1, 2}` và bằng `0` với `r = 0`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (`n <= 50`).

| Cách | Time | Space |
|---|---|---|
| 1. Thử `d = 0, 1, ...` | `O(n)` (hằng số nhỏ) | `O(1)` |
| 2. `min(r, 3 - r)` | `O(n)` | `O(1)` |
| 3. Đếm `x % 3 != 0` | `O(n)` | `O(1)` |
| 4. Đếm lớp số dư | `O(n)` | `O(1)` |
| 5. Bội gần nhất | `O(n)` | `O(1)` |

Không thể nhanh hơn `O(n)` vì phải nhìn mọi phần tử ít nhất một lần.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [18]:
sol = Solution()
impls = {
    "search": minimum_operations_search,
    "min": minimum_operations_min,
    "count": minimum_operations_count,
    "classes": minimum_operations_classes,
    "nearest": minimum_operations_nearest,
    "leetcode": sol.minimumOperations,
}


def check_all(nums, expected, label):
    for key, fn in impls.items():
        actual = fn(list(nums))
        assert actual == expected, (label, key, nums, actual, expected)
    print("PASS", label)


for idx, (nums, expected) in enumerate(examples, 1):
    check_all(nums, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

In [19]:
check_all([1], 1, "một phần tử dư 1")
check_all([2], 1, "một phần tử dư 2")
check_all([3], 0, "một phần tử đã chia hết")
check_all([50], 1, "giá trị lớn nhất, dư 2")
check_all([49], 1, "49 dư 1")
check_all([48], 0, "48 chia hết 3")
check_all([1, 1, 1, 1], 4, "toàn dư 1")
check_all([2, 2, 2, 2], 4, "toàn dư 2")
check_all([3, 6, 9, 12, 15], 0, "toàn bội của 3")
check_all([1, 2], 2, "cả hai đều cần 1 thao tác")
check_all([4, 5, 6], 2, "hai không chia hết, một chia hết")
check_all(list(range(1, 51)), 50 - 16, "1..50: có 16 bội của 3 (3, 6, ..., 48)")
check_all([1] * 50, 50, "50 phần tử dư 1")
check_all([3] * 50, 0, "50 phần tử bằng 3")

PASS một phần tử dư 1
PASS một phần tử dư 2
PASS một phần tử đã chia hết
PASS giá trị lớn nhất, dư 2
PASS 49 dư 1
PASS 48 chia hết 3
PASS toàn dư 1
PASS toàn dư 2
PASS toàn bội của 3
PASS cả hai đều cần 1 thao tác
PASS hai không chia hết, một chia hết
PASS 1..50: có 16 bội của 3 (3, 6, ..., 48)
PASS 50 phần tử dư 1
PASS 50 phần tử bằng 3


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **cộng số dư** `x % 3` làm chi phí (số dư `2` bị tính `2` thao tác); (b) **chỉ cho phép cộng** (bỏ quên thao tác trừ).

In [20]:
def wrong_sum_remainder(nums):
    return sum(x % 3 for x in nums)                  # SAI: số dư 2 tốn 1 chứ không phải 2


def wrong_only_add(nums):
    return sum((3 - x % 3) % 3 for x in nums)        # SAI: chỉ cộng -> số dư 1 phải cộng 2 lần


print(wrong_sum_remainder([2]), "!=", Solution().minimumOperations([2]))
print(wrong_only_add([1, 2, 3, 4]), "!=", Solution().minimumOperations([1, 2, 3, 4]))
assert wrong_sum_remainder([2]) != Solution().minimumOperations([2])
assert wrong_only_add([1, 2, 3, 4]) != Solution().minimumOperations([1, 2, 3, 4])

2 != 1
5 != 3


### 12.3. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc

Sinh `nums` có độ dài `1..50`, giá trị `1..50`. Ngoài ra, ta kiểm chứng **độc lập** bằng cách thử **mọi** giá trị đích (mọi bội `3` trong khoảng rộng): chi phí của `x` là `min(|x - t|)` trên mọi bội `t` của `3` (kể cả `t = 0`, `-3`, ...). Cuối cùng vét cạn **mọi** mảng độ dài `<= 4` với giá trị `1..9`.

In [21]:
def independent_reference(nums):
    targets = range(-6, 60, 3)                       # mọi bội của 3 trong khoảng rộng
    return sum(min(abs(x - t) for t in targets) for x in nums)


rng = random.Random(3190)
for trial in range(3000):
    n = rng.randint(1, 50)
    nums = [rng.randint(1, 50) for _ in range(n)]
    assert 1 <= len(nums) <= 50 and all(1 <= x <= 50 for x in nums)
    ref = independent_reference(nums)
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
    assert ref == len(nums) - sum(x % 3 == 0 for x in nums)
print("PASS 3000 mảng ngẫu nhiên đúng ràng buộc")

import itertools
cnt_all = 0
for n in range(1, 5):
    for nums in itertools.product(range(1, 10), repeat=n):
        ref = independent_reference(list(nums))
        for key, fn in impls.items():
            assert fn(list(nums)) == ref, (key, nums)
        cnt_all += 1
print("PASS vét cạn", cnt_all, "mảng độ dài <= 4 với giá trị 1..9")

PASS 3000 mảng ngẫu nhiên đúng ràng buộc


PASS vét cạn 7380 mảng độ dài <= 4 với giá trị 1..9


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cộng số dư `x % 3` làm chi phí

In [22]:
def sai_cong_so_du(nums):
    total = 0
    for x in nums:
        total += x % 3                   # SAI: số dư 2 phải tốn 1 (cộng 1), không phải 2
    return total


print("sai:", sai_cong_so_du([2, 2, 2]), "| đúng:", Solution().minimumOperations([2, 2, 2]))

sai: 6 | đúng: 3


**Vì sao sai:** số dư `x % 3` cho biết cần **trừ** bao nhiêu để về bội của `3` bên dưới; nhưng với số dư `2` thì **cộng 1** lên bội bên trên rẻ hơn. **Cách sửa:** `min(r, 3 - r)` hoặc đơn giản `1` cho mỗi phần tử không chia hết.

### Lỗi 2: Đếm nhầm các phần tử **chia hết** (ngược điều kiện)

In [23]:
def sai_dem_nguoc(nums):
    return sum(x % 3 == 0 for x in nums)           # SAI: đây là số phần tử ĐÃ chia hết, không cần thao tác


print("sai:", sai_dem_nguoc([1, 2, 3, 4]), "| đúng:", Solution().minimumOperations([1, 2, 3, 4]))

sai: 1 | đúng: 3


**Vì sao sai:** ta cần số phần tử **chưa** chia hết (`!= 0`). Ví dụ 1: có `3` phần tử chưa chia hết nhưng chỉ `1` phần tử chia hết. **Cách sửa:** `x % 3 != 0`; hoặc `len(nums) - số_chia_hết`.

### Lỗi 3: Nghĩ rằng một thao tác tác động lên nhiều phần tử

In [24]:
def sai_mot_thao_tac_nhieu_phan_tu(nums):
    # SAI: tưởng rằng "cộng 1 cho cả mảng" chỉ tính 1 thao tác
    best = len(nums)
    for delta in (-1, 0, 1):
        remain = sum((x + delta) % 3 != 0 for x in nums)
        best = min(best, (1 if delta != 0 else 0) + remain)
    return best


print("sai:", sai_mot_thao_tac_nhieu_phan_tu([1, 1, 1, 1]), "| đúng:", Solution().minimumOperations([1, 1, 1, 1]))

sai: 1 | đúng: 4


**Vì sao sai:** đề nói mỗi thao tác chỉ đụng **một** phần tử. Cộng/trừ `1` cho **cả mảng** tốn `n` thao tác, không phải `1`. Với `[1, 1, 1, 1]` đáp án đúng là `4`, bản sai nhầm là `1`. **Cách sửa:** chi phí là **tổng** theo từng phần tử.

### Lỗi 4: Cộng cả chi phí trừ lẫn chi phí cộng cho cùng một phần tử

In [25]:
def sai_tinh_ca_hai_huong(nums):
    total = 0
    for x in nums:
        r = x % 3
        total += r + (3 - r) % 3         # SAI: cộng cả chi phí trừ (r) lẫn chi phí cộng (3 - r)
    return total


print("sai:", sai_tinh_ca_hai_huong([1, 2, 3, 4]), "| đúng:", Solution().minimumOperations([1, 2, 3, 4]))

sai: 9 | đúng: 3


**Vì sao sai:** với mỗi phần tử ta chỉ **chọn một hướng** (trừ **hoặc** cộng), tức lấy `min`, không cộng cả hai. Bản sai cho `9` thay vì `3`. **Cách sửa:** `min(r, 3 - r)`, hoặc nhớ gọn là `1` cho mỗi phần tử không chia hết.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Cộng/trừ `1` cho **từng phần tử** để thành bội của `k`" ⇒ chi phí mỗi phần tử là `min(r, k - r)` với `r = x % k`.
- Với `k = 3`, mọi phần tử không chia hết đều tốn đúng `1` ⇒ **đếm** `x % 3 != 0`.
- Thao tác tác động **một** phần tử ⇒ tổng chi phí = **tổng** chi phí từng phần tử (độc lập).
- Nhớ `True == 1` để dùng `sum(điều_kiện for ...)` đếm gọn.
- Khi đề cho `n`, `nums[i]` nhỏ, hãy **tự kiểm chứng** công thức bằng vét cạn ngắn (như mục 12.3).
- Bài họ hàng: "số thao tác để tất cả bằng nhau", "để tổng chia hết cho `k`".

## 15. Biến thể của bài

1. **Chia hết cho `k` bất kỳ:** `sum(min(x % k, k - x % k) for x in nums)`.
2. **Chỉ được cộng `1`:** chi phí mỗi phần tử là `(k - x % k) % k`.
3. **Chỉ được trừ `1`:** chi phí mỗi phần tử là `x % k` (và phải `>= 0`).
4. **Phí khác nhau:** cộng tốn `a`, trừ tốn `b`: chi phí mỗi phần tử là `min(r * b, (k - r) * a)`.
5. **Minimum Moves to Equal Array Elements II (462)**, **Greatest Sum Divisible by Three (1262)**, **Minimum Moves to Equal Array Elements (453)**: cùng họ chi phí theo số dư/khoảng cách.

In [26]:
def ops_divisible_by_k(nums, k):
    return sum(min(x % k, k - x % k) for x in nums)


def ops_only_add(nums, k):
    return sum((k - x % k) % k for x in nums)


def ops_only_subtract(nums, k):
    return sum(x % k for x in nums)


def ops_weighted(nums, k, add_cost, sub_cost):
    return sum(min((x % k) * sub_cost, ((k - x % k) % k) * add_cost) for x in nums)


print(ops_divisible_by_k([1, 2, 3, 4, 7], 5))                     # chi phí 1 + 2 + 2 + 1 + 2 = 8
print(ops_only_add([1, 2, 3, 4], 3))                              # 2 + 1 + 0 + 2 = 5
print(ops_only_subtract([1, 2, 3, 4], 3))                         # 1 + 2 + 0 + 1 = 4
print(ops_weighted([1, 2, 3, 4], 3, add_cost=1, sub_cost=5))      # 1: cộng 2 lần (2), 2: cộng 1 lần (1), 3: 0, 4: cộng 2 lần (2) -> 5
assert ops_divisible_by_k([1, 2, 3, 4], 3) == Solution().minimumOperations([1, 2, 3, 4])
assert ops_divisible_by_k([1, 2, 3, 4, 7], 5) == 8
assert ops_only_add([1, 2, 3, 4], 3) == 5
assert ops_only_subtract([1, 2, 3, 4], 3) == 4
assert ops_weighted([1, 2, 3, 4], 3, add_cost=1, sub_cost=5) == 5

8
5
4
5


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử `d = 0, 1, ...` | tìm khoảng dịch nhỏ nhất cho từng phần tử | `O(n)` | `O(1)` | khi chưa thấy công thức, dễ chứng minh |
| 2. `min(r, 3 - r)` | trừ hoặc cộng, chọn rẻ hơn | `O(n)` | `O(1)` | tổng quát cho bội `k` |
| 3. Đếm `x % 3 != 0` | mỗi phần tử không chia hết tốn 1 | `O(n)` | `O(1)` | **nên nộp** |
| 4. Đếm lớp số dư | `cnt[1] + cnt[2]` | `O(n)` | `O(1)` | khi phí mỗi lớp khác nhau |
| 5. Bội gần nhất | `abs(x - ((x+1)//3)*3)` | `O(n)` | `O(1)` | nhấn mạnh ý nghĩa khoảng cách |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `17 % 3`, `min(2, 3 - 2)`, `sum(x % 3 != 0 for x in [3, 4, 5, 6])`, `Solution().minimumOperations([7, 8, 9])`, `Solution().minimumOperations([30])`.

**Bài 2 (tính tay).** Với `nums = [10, 11, 12, 13, 14]`: lập bảng `x`, `x % 3`, chi phí, rồi tổng.

**Bài 3 (điền TODO).** Hoàn thành `minimum_operations_todo(nums)` theo Cách 2 (`min(r, 3 - r)`).

In [27]:
def minimum_operations_todo(nums):
    total = 0
    # TODO 1: duyệt từng x trong nums, tính r = x % 3
    # TODO 2: cộng vào total giá trị min(r, 3 - r)
    # TODO 3: trả về total
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `minimum_operations_nearest2(nums)` theo Cách 5: với mỗi `x`, tính bội của `3` gần nhất **bằng `round`** hoặc cách khác **không** dùng công thức `((x + 1) // 3) * 3`, rồi cộng khoảng cách.

**Bài 5 (biến thể / mini challenge).** (a) Viết `minimum_operations_k(nums, k)` cho bội `k` bất kỳ. (b) Viết `final_array(nums)` trả về **mảng sau khi biến đổi** (mỗi phần tử thành bội của `3` gần nhất; nếu có hai bội cách đều thì chọn nhỏ hơn, nhưng với `k = 3` không bao giờ xảy ra), và chứng minh số thao tác bằng tổng `|x - y|`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `2`, `1`, `3`, `2` (`7` dư 1, `8` dư 2, `9` chia hết), `0`. Bài 3: `r = x % 3`, `total += min(r, 3 - r)`.
- **Mức 2 (vừa):** Bài 2: `10→1 (dư 1)`, `11→1 (dư 2)`, `12→0`, `13→1`, `14→1` ⇒ tổng `4`. Bài 4: với `x % 3 == 1` bội gần nhất là `x - 1`, với `x % 3 == 2` là `x + 1`; hoặc `round(x / 3) * 3` (an toàn vì `x / 3` không bao giờ có phần thập phân đúng `.5`).
- **Mức 3 (sâu):** Bài 5(a): `sum(min(x % k, k - x % k) for x in nums)`. Bài 5(b): `y = ((x + 1) // 3) * 3`; tổng `abs(x - y)` chính là đáp án của bài gốc.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert 17 % 3 == 2
assert min(2, 3 - 2) == 1
assert sum(x % 3 != 0 for x in [3, 4, 5, 6]) == 2
assert Solution().minimumOperations([7, 8, 9]) == 2
assert Solution().minimumOperations([30]) == 0

# Bài 2
nums2 = [10, 11, 12, 13, 14]
assert [x % 3 for x in nums2] == [1, 2, 0, 1, 2]
assert Solution().minimumOperations(nums2) == 4


# Bài 3
def minimum_operations_todo_ans(nums):
    total = 0
    for x in nums:
        r = x % 3
        total += min(r, 3 - r)
    return total


for nums, expected in examples:
    assert minimum_operations_todo_ans(nums) == expected


# Bài 4
def minimum_operations_nearest2(nums):
    return sum(abs(x - round(x / 3) * 3) for x in nums)


for nums, expected in examples:
    assert minimum_operations_nearest2(nums) == expected
for x in range(1, 51):
    assert minimum_operations_nearest2([x]) == Solution().minimumOperations([x])


# Bài 5(a)
def minimum_operations_k(nums, k):
    return sum(min(x % k, k - x % k) for x in nums)


assert minimum_operations_k([1, 2, 3, 4], 3) == 3
assert minimum_operations_k([1, 2, 3, 4, 7], 5) == 8
assert minimum_operations_k([5, 10, 15], 5) == 0


# Bài 5(b)
def final_array(nums):
    return [((x + 1) // 3) * 3 for x in nums]


assert final_array([1, 2, 3, 4]) == [0, 3, 3, 3]
for nums in ([1, 2, 3, 4], [5, 7, 9], [50, 49, 48]):
    y = final_array(nums)
    assert all(v % 3 == 0 for v in y)
    assert sum(abs(a - b) for a, b in zip(nums, y)) == Solution().minimumOperations(nums)
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mỗi phần tử không chia hết cho `3` tốn đúng một thao tác)

```python
class Solution:
    def minimumOperations(self, nums: List[int]) -> int:
        return sum(x % 3 != 0 for x in nums)
```

**Checklist**

- [ ] Hiểu mỗi thao tác chỉ đụng **một** phần tử (nên chi phí cộng dồn theo từng phần tử)?
- [ ] Số dư `1` ⇒ trừ `1`; số dư `2` ⇒ cộng `1`; số dư `0` ⇒ không làm gì?
- [ ] Đếm các phần tử **không** chia hết (`!= 0`), không phải chia hết?
- [ ] Không cộng thẳng `x % 3` (số dư `2` tốn `1`, không phải `2`)?
- [ ] Đã thử `[1,2,3,4]` (`3`), `[3,6,9]` (`0`), `[1]` (`1`), `[2]` (`1`)?
- [ ] Skeleton đúng `class Solution` / `def minimumOperations(self, nums: List[int]) -> int`?